# FLAG 2027 — NB-5: external identities (MAV-Celeb v1-v3) for the face-voice bridge

Every encoder swap failed (ArcFace, ReDimNet2): the bottleneck is the cross-modal bridge learned from only
70 speakers. External data is allowed if declared, so this notebook adds the organisers' own earlier releases:

| source | languages | ids | wav | size |
|---|---|---:|---:|---:|
| `v3_train` | English + German | 50 | 3.0k | 1.6 GB |
| `v1_train` | English + Urdu | 64 | 18.0k | 11 GB |
| `v2_complete` | English + Hindi | 84 | 20.7k | 13 GB |

No Bangla anywhere (FAME rule: no pretraining on the unheard language).
Features are extracted with the organisers' exact encoders (verified cos 1.0000 on v4 train):
voice `yangwang825/ecapa-tdnn-vox2` (192-d), face VGGFace fc7 (4096-d). 8 frames per video.

**Settings:** GPU T4, **Internet ON** (Drive + HF + VGGFace weights). **Inputs:** `flag2027-features`,
`flag2027-feats-v2`. Zips are downloaded one at a time to local disk and deleted after extraction.
Expected: download + extraction ~40-70 min (Drive speed dominates), training ~40 min.

Output: `feats_ext/` (features + meta, ~0.3 GB), `overlap.csv`, `ablation_ext.csv`, `submissions/ext_*.zip`
+ `*_scores.npz` + `*_matrix.npz`.

In [1]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

KAGGLE | input /kaggle/input | working /kaggle/working


In [2]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_extract.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
open('flag_v2.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
print('written:', ['flag_lib.py', 'flag_extract.py', 'flag_v2.py'])

written: ['flag_lib.py', 'flag_extract.py', 'flag_v2.py']


## 1. Download + extract (organiser encoders), one zip at a time

In [3]:
if not LOCAL:
    !pip -q install -U "speechbrain>=1.1.1" 2>&1 | tail -2
import shutil, time, numpy as np, pandas as pd, torch
import flag_extract as X
OUTX = WORKDIR / "feats_ext"; OUTX.mkdir(exist_ok=True)
TMP = Path(os.environ.get("FLAG_TMP", "/tmp/flag_ext")); TMP.mkdir(parents=True, exist_ok=True)
print("free disk at", TMP, round(shutil.disk_usage(TMP).free / 1e9, 1), "GB")
ve = X.OrganiserVoice(WORKDIR / "_ecapa_btc")
fe = X.VGGFace(WORKDIR / "_vggface")
SOURCES = {"mini_v3": None} if LOCAL else X.EXT_SOURCES        # local smoke test uses a prepared mini zip
for name, spec in SOURCES.items():
    if (OUTX / f"ext_{name}_voice.npy").exists() and (OUTX / f"ext_{name}_face.npy").exists():
        print("skip (exists)", name); continue
    t0 = time.time()
    try:
        if LOCAL:
            zp = Path(os.environ["FLAG_EXT_MINI"])
        else:
            fid, gb = spec
            zp = X.fetch_zip(fid, TMP / f"{name}.zip", min_free_gb=gb * 1.2 + 2)
        X.extract_external(zp, name, OUTX, ve, fe, frames_per_video=8)
    except (AssertionError, RuntimeError, OSError) as e:        # Drive quota / disk: skip this source, keep going
        print(f"SKIPPED {name}: {type(e).__name__}: {e}")
        continue
    finally:
        if not LOCAL and (TMP / f"{name}.zip").exists():
            (TMP / f"{name}.zip").unlink()
    print(f"{name}: {time.time() - t0:.0f}s", flush=True)
del ve, fe; torch.cuda.empty_cache()
X.write_manifest(OUTX, dict(sources=list(SOURCES), encoders="organiser VGGFace fc7 + yangwang825/ecapa-tdnn-vox2"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 34.4 MB/s eta 0:00:00
free disk at /tmp/flag_ext 1201.7 GB


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/24.9M [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


03:30:40 downloading vgg_face_dag.py
03:30:41 downloading vgg_face_dag.pth
03:31:02 downloaded v3_train.zip: 1.69 GB, 45683 entries
03:31:03 v3_train: 3017 wav, 41650 jpg in 357 videos, 50 ids, langs ['english', 'german']
03:31:05   voice v3_train 1/3017  2s  eta 6862s  oom=0
03:31:12   voice v3_train 128/3017  10s  eta 223s  oom=0
03:31:21   voice v3_train 535/3017  19s  eta 87s  oom=0
03:31:30   voice v3_train 1395/3017  28s  eta 32s  oom=0
03:31:38   voice v3_train 2940/3017  36s  eta 1s  oom=0
03:31:38   voice v3_train 3017/3017  36s  eta 0s  oom=0
03:31:39   face v3_train 0/2791
03:31:57 v3_train done: (3017, 192) (2791, 4096)
v3_train: 72s
03:34:35 downloaded v1_train.zip: 11.30 GB, 298769 entries
03:34:38 v1_train: 18043 wav, 278610 jpg in 865 videos, 64 ids, langs ['english', 'urdu']
SKIPPED v1_train: AssertionError: ('voices/id0017/Urdu/KoL1WcDzTOc/00001.wav', 44100)
03:34:38 attempt 0: not a zip yet: b'<!DOCTYPE html><html><head><title>Google Drive - Quota exceeded</title><me

{'created': '2026-09-26 03:34',
 'files': {'ext_v3_train_face.npy': [2791, 4096],
  'ext_v3_train_voice.npy': [3017, 192]},
 'sources': ['v3_train', 'v1_train', 'v2_complete'],
 'encoders': 'organiser VGGFace fc7 + yangwang825/ecapa-tdnn-vox2'}

## 2. Identity overlap with v4 train/dev -> exclusion list

In [4]:
# Identity overlap: an external speaker who is also a v4 train/dev person would leak. Face centroids (VGG fc7,
# standardised with v4-train stats, PCA-256) per identity; threshold calibrated on v4 train: two disjoint halves
# of the SAME v4 speaker vs DIFFERENT v4 speakers.
os.environ["FLAG_DATA"] = str(IN)
os.environ["FLAG_FEATS_EXTRA"] = str(OUTX)
import flag_v2 as V
store = V.FeatureStore(IN)
Xf = store.Xf; mf, sf = Xf.mean(0), Xf.std(0) + 1e-6
P = np.linalg.svd((Xf - mf) / sf, full_matrices=False)[2][:256].T
emb = lambda A: V.l2n(((A - mf) / sf) @ P)
rng = np.random.RandomState(0)
same, diff, cent = [], [], {}
for s_ in np.unique(store.spk):
    i = np.where(store.spk == s_)[0]
    if len(i) < 8: continue
    rng.shuffle(i); h = len(i) // 2
    a, b = V.l2n(emb(Xf[i[:h]]).mean(0, keepdims=True))[0], V.l2n(emb(Xf[i[h:]]).mean(0, keepdims=True))[0]
    same.append(a @ b); cent[s_] = V.l2n(emb(Xf[i]).mean(0, keepdims=True))[0]
C4 = np.stack(list(cent.values())); diff = (C4 @ C4.T)[np.triu_indices(len(C4), 1)]
THR = float(max(np.percentile(diff, 99.9), np.percentile(same, 1)))
print(f"v4 train: same-person half-centroid cos p1 {np.percentile(same, 1):.3f} | different-person p99.9 {np.percentile(diff, 99.9):.3f} -> threshold {THR:.3f}")
devF = np.concatenate([store.dev[k][0] for k in V.CELLS]); devE = emb(devF)
rows = []
for f in sorted(OUTX.glob("ext_*_face.npy")):
    src = f.name[4:-9]; fm = pd.read_csv(OUTX / f"ext_{src}_face_meta.csv"); A = np.load(f).astype(np.float32)
    E_ = emb(A)
    for s_, g in fm.groupby("spk"):
        c = V.l2n(E_[g.index].mean(0, keepdims=True))[0]
        best_tr = C4 @ c
        nn_dev = np.sort(devE @ c)[::-1][:20].mean()       # centroid vs its 20 nearest dev faces
        rows.append(dict(src=src, spk=s_, key=f"{src}:{s_}", max_cos_v4train=round(float(best_tr.max()), 3),
                         v4train_id=list(cent)[int(best_tr.argmax())], top20_cos_dev=round(float(nn_dev), 3)))
OV = pd.DataFrame(rows)
OV["overlap"] = (OV.max_cos_v4train > THR) | (OV.top20_cos_dev > THR)
OV.to_csv(WORKDIR / "overlap.csv", index=False)
EXCLUDE = tuple(OV[OV.overlap].key)
print(OV.sort_values("max_cos_v4train", ascending=False).head(12).to_string(index=False))
print(f"excluded {len(EXCLUDE)} of {len(OV)} external identities:", EXCLUDE)

03:36:21 feature dir: /kaggle/input/datasets/chnnhnnguyn/flag2027-feats-v2/feats | 33 arrays
03:36:21 feature dir: /kaggle/working/feats_ext | 2 arrays
v4 train: same-person half-centroid cos p1 0.874 | different-person p99.9 0.493 -> threshold 0.874
     src    spk             key  max_cos_v4train v4train_id  top20_cos_dev  overlap
v3_train id0042 v3_train:id0042            0.554      id034          0.593    False
v3_train id0026 v3_train:id0026            0.492      id065          0.500    False
v3_train id0049 v3_train:id0049            0.487      id034          0.553    False
v3_train id0034 v3_train:id0034            0.483      id049          0.416    False
v3_train id0021 v3_train:id0021            0.465      id049          0.409    False
v3_train id0030 v3_train:id0030            0.456      id062          0.438    False
v3_train id0001 v3_train:id0001            0.446      id034          0.523    False
v3_train id0009 v3_train:id0009            0.409      id034          0.527   

## 3. Internal ablation (v4 speaker-disjoint), external rows in training only

In [5]:
# Internal validation = v4 speaker-disjoint splits (external rows are always TRAINING only). One variable at a time.
REC = dict(V.BASE, head="mlp", drop=0.3, emb=128, epochs=40)           # EXP-007 recipe
ALL = tuple(s for s in (["mini_v3"] if LOCAL else list(X.EXT_SOURCES)) if (OUTX / f"ext_{s}_voice.npy").exists())
print("external sources available:", ALL)
KW = dict(seeds=(1,), n_models=1) if LOCAL else dict(seeds=(1, 2, 3), n_models=3)
if LOCAL:
    REC["epochs"] = 2
T = V.Table(WORKDIR / "ablation_ext.csv")
T.run(store, "EXT", "A control (v4 only)", REC, **KW)
if not LOCAL and "v3_train" in ALL:
    T.run(store, "EXT", "B + v3 (En/German)", dict(REC, ext=("v3_train",), ext_exclude=EXCLUDE), **KW)
T.run(store, "EXT", "C + all, cap 150/spk", dict(REC, ext=ALL, ext_exclude=EXCLUDE), **KW)
T.run(store, "EXT", "D + all, cap 400/spk", dict(REC, ext=ALL, ext_cap=400, ext_exclude=EXCLUDE), **KW)
T.run(store, "EXT", "E + all, cap 150, 20 ep", dict(REC, ext=ALL, epochs=2 if LOCAL else 20, ext_exclude=EXCLUDE), **KW)
print(T.df[["run", "int_ng", "int_g", "proxy", "sd_g", "sec"]].to_string(index=False))

external sources available: ('v3_train',)
03:47:30 [EXT] A control (v4 only)                           ng  21.83  g  30.49  proxy  26.16  (227s)
03:47:30 ext rows ('v3_train',): 3006 rows, 50 speakers (cap 150, excluded 0)
03:50:12 [EXT] B + v3 (En/German)                            ng  22.64  g  32.71  proxy  27.68  (162s)
03:51:03 [EXT] C + all, cap 150/spk                          ng  22.64  g  32.71  proxy  27.68  (51s)
03:51:03 ext rows ('v3_train',): 3017 rows, 50 speakers (cap 400, excluded 0)
03:53:45 [EXT] D + all, cap 400/spk                          ng  22.56  g  32.86  proxy  27.71  (162s)
03:54:15 [EXT] E + all, cap 150, 20 ep                       ng  22.81  g  32.92  proxy  27.87  (30s)
                    run  int_ng  int_g  proxy  sd_g  sec
    A control (v4 only)   21.83  30.49  26.16  2.21  227
     B + v3 (En/German)   22.64  32.71  27.68  0.73  162
   C + all, cap 150/spk   22.64  32.71  27.68  0.73   51
   D + all, cap 400/spk   22.56  32.86  27.71  0.80  162
E + 

## 4. Dev scores for local selection

In [6]:
# Dev scores for the control and every external variant (the local selector decides; nothing is chosen here).
import json
SUBS = WORKDIR / "submissions"; SUBS.mkdir(exist_ok=True)
N_MODELS = 2 if LOCAL else 10
RUNS = {"ext_A_control": REC,
        "ext_C_all150": dict(REC, ext=ALL, ext_exclude=EXCLUDE),
        "ext_D_all400": dict(REC, ext=ALL, ext_cap=400, ext_exclude=EXCLUDE)}
for name, cfg in RUNS.items():
    out = SUBS / f"{name}.zip"
    if out.exists():
        print("skip (exists)", out.name); continue
    t0 = time.time()
    sc = V.dev_scores(store, cfg, n_models=N_MODELS, cca_w=0.25, matrix_path=SUBS / f"{name}_matrix.npz")
    np.savez(SUBS / f"{name}_scores.npz", **{"/".join(k): v for k, v in sc.items()})
    V.write_zip(store, out, sc); V.check_zip(store, out)
    pd.DataFrame([dict(zip=out.name, sec=round(time.time() - t0), cfg=json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}))]).to_csv(
        SUBS / "runs.csv", mode="a", header=not (SUBS / "runs.csv").exists(), index=False)
    print(f"{name}: {time.time() - t0:.0f}s", flush=True)
for p in [WORKDIR / "_ecapa_btc", WORKDIR / "_vggface"]:
    shutil.rmtree(p, ignore_errors=True)
print("Download: submissions/, feats_ext/, overlap.csv, ablation_ext.csv")

03:58:35 zip OK: /kaggle/working/submissions/ext_A_control.zip
ext_A_control: 260s
04:03:14 zip OK: /kaggle/working/submissions/ext_C_all150.zip
ext_C_all150: 279s
04:07:52 zip OK: /kaggle/working/submissions/ext_D_all400.zip
ext_D_all400: 278s
Download: submissions/, feats_ext/, overlap.csv, ablation_ext.csv


In [8]:
import shutil

# Đường dẫn tới thư mục bạn muốn nén
thu_muc_can_nen = "/kaggle/working/submissions"

# Tên file zip mong muốn (không cần điền đuôi .zip)
ten_file_zip = "submissions"

# Thực hiện nén thành file zip
shutil.make_archive(ten_file_zip, 'zip', thu_muc_can_nen)
print(f"Đã nén thành công thành {ten_file_zip}.zip")


Đã nén thành công thành submissions.zip
